In [3]:
import sys
import os
sys.path.append(os.path.abspath('../../src/stream_1'))
import pandas as pd
from bounded_regression import LogisticBoroughForecaster

dfs = []
for target, label in [('active', '% Active'), ('VolAny', '% Volunteering')]:
    df = LogisticBoroughForecaster(target_col=target).fit_predict()
    df['metric'] = label
    dfs.append(df)

out = pd.concat(dfs)
out['period'] = out['year'].apply(lambda y: 'forecast' if y >= 2024 else 'historical')
os.makedirs('../../visualisation/data', exist_ok=True)
out.to_csv('../../visualisation/data/tableau_borough_logistic.csv', index=False)

/Users/josh/Library/CloudStorage/GoogleDrive-jeverer@gmail.com/My Drive/London Sport/london_sport2/src/loading_data/load_data.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['LOG_MEMS7_ALL'] = np.log1p(df['MEMS7_ALL'])


DataFrame Cleaned Successfully...
DataFrame Information:
>>> Columns:
Index(['LA_2023', 'LondInOut', 'MEMS7_ALL', 'VolAny', 'LOG_MEMS7_ALL',
       'active', 'year', 'month', 'LA_Name'],
      dtype='str')
>>> Shape (117679, 9)


/Users/josh/Library/CloudStorage/GoogleDrive-jeverer@gmail.com/My Drive/London Sport/london_sport2/src/loading_data/load_data.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['LOG_MEMS7_ALL'] = np.log1p(df['MEMS7_ALL'])


DataFrame Cleaned Successfully...
DataFrame Information:
>>> Columns:
Index(['LA_2023', 'LondInOut', 'MEMS7_ALL', 'VolAny', 'LOG_MEMS7_ALL',
       'active', 'year', 'month', 'LA_Name'],
      dtype='str')
>>> Shape (117679, 9)


In [4]:
import pandas as pd
import numpy as np
from scipy.special import logit, expit
from sklearn.linear_model import LinearRegression

class_labels = {
    0: 'Middle-aged Working Fathers', 1: 'Highly Educated Working Mothers',
    2: 'Professional Fathers', 3: 'Part-time Working Professional Mothers',
    4: 'Later-career Middle Class Workers', 5: 'Highly Educated Early Retirees',
    6: 'Professional Women from Ethnic Minority Backgrounds', 7: 'British-born Retirees',
    8: 'Students in Shared Housing', 9: 'Young Middle Class Workers',
    10: 'Motivated Young Professional Men', 11: 'Graduate Professionals in Shared Housing',
    12: 'Later-career Professional Women', 13: 'Unemployed Adults from Deprived Backgrounds',
    14: 'Long-term Sick and Disabled Adults', 15: 'Mid-career Professionals Living Alone',
    16: 'Mothers and Carers from Deprived Backgrounds', 17: 'Young Professional Women',
    18: 'Older Parents Approaching Retirement', 19: 'Independent Older Retirees',
    20: 'International Educated Professionals', 21: 'Young Students Living with Parents',
    22: 'Older Retirees with Fewer Qualifications', 23: 'Later-career Professional Men',
    24: 'Disabled Older Retirees', 25: 'Adults from Deprived Backgrounds Living with Parents',
    26: 'Young Professionals Living with Parents'
}

TRAIN_YEARS = [2017, 2018, 2019, 2020, 2021, 2022, 2023]
FORE_YEARS  = [2024, 2025, 2026, 2027]

df = pd.read_csv('../../data/master_data/latent_class_forecasting_data.csv')
df = df.rename(columns={'calendar_year': 'year'})
df = df[df['year'].isin(TRAIN_YEARS)]

rows = []
for cls in df['LCA_Class'].unique():
    grp = df[df['LCA_Class'] == cls].sort_values('year')
    Y_train = grp['Mean_active'].values
    X_train = (np.array(TRAIN_YEARS) - 2017).reshape(-1, 1)
    X_fore  = (np.array(FORE_YEARS)  - 2017).reshape(-1, 1)
    X_full  = np.concatenate([X_train, X_fore])

    MIN = Y_train.min() - np.std(Y_train)
    MAX = Y_train.max() + np.std(Y_train)
    MIN = max(MIN, 0.001)
    MAX = min(MAX, 0.999)

    Y_scaled = (Y_train - MIN) / (MAX - MIN)
    Y_scaled = np.clip(Y_scaled, 0.001, 0.999)

    model = LinearRegression().fit(X_train, logit(Y_scaled))
    Y_pred_full = (MIN + expit(model.predict(X_full)) * (MAX - MIN)) * 100
    Y_fore_vals = (MIN + expit(model.predict(X_fore))  * (MAX - MIN)) * 100
    Y_target    = np.concatenate([Y_train * 100, Y_fore_vals])

    for year, target, predicted in zip(TRAIN_YEARS + FORE_YEARS, Y_target, Y_pred_full):
        rows.append({
            'LCA_Class': int(cls),
            'LCA_Label': class_labels[int(cls)],
            'year': year,
            'target': round(target, 2),
            'predicted': round(predicted, 2),
            'period': 'forecast' if year >= 2024 else 'historical'
        })

os.makedirs('../../visualisation/data', exist_ok=True)
pd.DataFrame(rows).to_csv('../../visualisation/data/tableau_cluster_logistic.csv', index=False)